# DermAI V2: Full ISIC Dataset Training (Google Colab)

This notebook is designed to train the `DermAIEngine` on the massive 30GB+ ISIC skin lesion dataset using a Google Colab GPU.

### Prerequisites:
1. **Enable GPU:** In Colab, go to `Runtime` -> `Change runtime type` -> select `T4 GPU` or `A100 GPU`.
2. **Kaggle API Key:** You need a Kaggle account to download the ISIC 2020 dataset.
   - Go to Kaggle.com -> Settings -> Create New Token (downloads `kaggle.json`).
   - Run the first cell and upload `kaggle.json` when prompted.
3. **CRITICAL: Accept Competition Rules (Fixes 403 Error)**
   - You **MUST** manually visit the competition page: [ISIC 2020 Melanoma Classification](https://www.kaggle.com/c/isic-2020-melanoma-classification/rules)
   - Click the **"I Understand and Accept"** button on the rules page.
   - If you do not do this, the API will return a `403 Client Error: Forbidden`.

In [ ]:
!pip install torch torchvision transformers pandas scikit-learn pillow kaggle -q

import os
from google.colab import files

print('Please upload your kaggle.json file:')
uploaded = files.upload()

!mkdir -p ~/.kaggle/ && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# Download the ISIC 2020 dataset (contains 33,000+ images and rich metadata)
!kaggle competitions download -c isic-2020-melanoma-classification
!unzip -q isic-2020-melanoma-classification.zip -d isic_data
print('Dataset downloaded and extracted successfully!')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from PIL import Image
from torchvision import transforms
from transformers import ViTModel, ViTConfig
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# 1. Re-define the DermAI Architecture
class MetadataCrossAttention(nn.Module):
    def __init__(self, embed_dim, metadata_dim):
        super().__init__()
        self.metadata_proj = nn.Linear(metadata_dim, embed_dim)
        self.query_proj = nn.Linear(embed_dim, embed_dim)
        self.key_proj = nn.Linear(embed_dim, embed_dim)
        self.value_proj = nn.Linear(embed_dim, embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads=8, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x, metadata):
        m_proj = self.metadata_proj(metadata).unsqueeze(1)
        q = self.query_proj(m_proj)
        k = self.key_proj(x)
        v = self.value_proj(x)
        attn_out, _ = self.attn(q, k, v)
        return self.norm(m_proj + attn_out).squeeze(1)

class DermAIEngine(nn.Module):
    def __init__(self, metadata_dim=4):
        super().__init__()
        self.config = ViTConfig.from_pretrained('google/vit-base-patch16-224-in21k')
        self.vit = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
        self.cross_attn = MetadataCrossAttention(self.config.hidden_size, metadata_dim)
        self.risk_head = nn.Linear(self.config.hidden_size, 1)
        self.mutation_head = nn.Linear(self.config.hidden_size, 1)

    def forward(self, pixel_values, metadata):
        patch_embeddings = self.vit(pixel_values=pixel_values).last_hidden_state
        fused = self.cross_attn(patch_embeddings, metadata)
        return torch.sigmoid(self.risk_head(fused)), torch.sigmoid(self.mutation_head(fused))


In [ ]:
# 2. Build the ISIC Dataloader
class ISICDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
        
        # Handle missing metadata
        self.df['age_approx'] = self.df['age_approx'].fillna(45.0)
        self.df['sex'] = self.df['sex'].fillna('unknown')
        self.df['anatom_site_general_challenge'] = self.df['anatom_site_general_challenge'].fillna('unknown')

    def __len__(self): 
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, f"{row['image_name']}.jpg")
        image = Image.open(img_path).convert('RGB')
        if self.transform: image = self.transform(image)
        
        # Encode Metadata
        age_norm = row['age_approx'] / 100.0
        sex_enc = 1.0 if row['sex'] == 'male' else 0.0
        loc_map = {'head/neck': 0.0, 'torso': 0.25, 'upper extremity': 0.5, 'lower extremity': 0.75, 'palms/soles': 1.0}
        loc_enc = loc_map.get(str(row['anatom_site_general_challenge']).lower(), 0.5)
        
        md = torch.tensor([age_norm, sex_enc, loc_enc, 0.0], dtype=torch.float32)
        risk = torch.tensor([row['target']], dtype=torch.float32)
        mut = torch.tensor([0.0], dtype=torch.float32) # ISIC lacks BRAF status
        
        return image, md, risk, mut

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

df = pd.read_csv('isic_data/train.csv')
train_df, val_df = train_test_split(df, test_size=0.1, stratify=df['target'])
train_df.to_csv('isic_data/train_split.csv', index=False)
val_df.to_csv('isic_data/val_split.csv', index=False)

train_loader = DataLoader(ISICDataset('isic_data/train_split.csv', 'isic_data/jpeg/train', transform), batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(ISICDataset('isic_data/val_split.csv', 'isic_data/jpeg/train', transform), batch_size=64, num_workers=2)


In [ ]:
# 3. Training Loop
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DermAIEngine(metadata_dim=4).to(device)

# Freeze the heavy ViT backbone initially to safely train our custom cross-attention
for param in model.vit.parameters(): 
    param.requires_grad = False

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
criterion = nn.BCELoss()

epochs = 10
for epoch in range(epochs):
    model.train()
    for i, (imgs, mds, risks, _) in enumerate(train_loader):
        optimizer.zero_grad()
        risk_pred, _ = model(imgs.to(device), mds.to(device))
        loss = criterion(risk_pred, risks.to(device))
        loss.backward()
        optimizer.step()
        
        if i % 100 == 0: 
            print(f"Epoch {epoch+1}/{epochs} | Batch {i}/{len(train_loader)} | Loss: {loss.item():.4f}")

    # Validation
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for imgs, mds, risks, _ in val_loader:
            risk_pred, _ = model(imgs.to(device), mds.to(device))
            preds.extend(risk_pred.cpu().numpy())
            labels.extend(risks.cpu().numpy())
            
    val_auc = roc_auc_score(labels, preds)
    print(f"\n>>> Epoch {epoch+1} Completed | Validation ROC-AUC: {val_auc:.4f} <<<\n")


In [ ]:
# 4. Save and Download Weights
save_path = 'engine_isic_full.pth'
torch.save(model.state_dict(), save_path)
print(f"Model saved to {save_path}")

from google.colab import files
files.download(save_path)
